In [ ]:
import glob
import numpy as np

import matplotlib.pyplot as plt
from scipy.ndimage import median_filter, gaussian_filter
from astropy.nddata import block_reduce
from astropy.stats import sigma_clip

CENTER_FREQ = 1420e6
OFFSET_FREQ = 1416e6
SAMPLE_RATE = 2.4e6
FFT_LEN = 4096

FREQ = (np.fft.fftshift(np.fft.fftfreq(FFT_LEN, 1 / SAMPLE_RATE)) + CENTER_FREQ) / 1e6
FREQ_OFF = (np.fft.fftshift(np.fft.fftfreq(FFT_LEN, 1 / SAMPLE_RATE)) + OFFSET_FREQ) / 1e6

List available on/off spectra

In [ ]:
import glob
filenames_on = glob.glob('raw/*_on.npy')
filenames_off = [filename.replace('_on', '_off') for filename in filenames_on]
len(filenames_on)

Load them in

In [ ]:
spectra_on = [np.load(filename) for filename in filenames_on]
spectra_off = [np.load(filename) for filename in filenames_off]

Plot first spectrum

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(3, 1, 1)
ax.set_xticks([])
ax.plot(FREQ, spectra_on[0])
ax = fig.add_subplot(3, 1, 2)
ax.set_xticks([])
ax.plot(FREQ_OFF, spectra_off[0])
ax = fig.add_subplot(3, 1, 3)
ax.plot(FREQ, spectra_on[0] - spectra_off[0])

These spectra have RFI spikes, let's try and mask these out with sigma clipping

In [ ]:
spectra_on = [sigma_clip(spec) for spec in spectra_on]
spectra_off = [sigma_clip(spec) for spec in spectra_off]

There is always a spike in the middle, so let's get rid of that even if sigma clipping didn't manage to:

In [ ]:
for spec in spectra_on:
    spec.mask[FFT_LEN // 2 - 19:FFT_LEN // 2 + 20] = True
for spec in spectra_off:
    spec.mask[FFT_LEN // 2 - 19:FFT_LEN // 2 + 20] = True

Let's plot again:

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(3, 1, 1)
ax.set_xticks([])
ax.plot(FREQ, spectra_on[0], lw=0.1)
ax = fig.add_subplot(3, 1, 2)
ax.plot(FREQ_OFF, spectra_off[0], lw=0.1)
ax.set_xticks([])
ax = fig.add_subplot(3, 1, 3)
ax.plot(FREQ, spectra_on[0] - spectra_off[0], lw=0.1)

Nice, now we can try and spectrally downsample this:

In [ ]:
DOWNSAMPLE = 10
freq_down = block_reduce(FREQ, DOWNSAMPLE, np.mean)
freq_down_off = block_reduce(FREQ_OFF, DOWNSAMPLE, np.mean)
spectra_on_down = [block_reduce(spec, DOWNSAMPLE, np.mean) for spec in spectra_on]
spectra_off_down = [block_reduce(spec, DOWNSAMPLE, np.mean) for spec in spectra_off]

And plot again:

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(3, 1, 1)
ax.plot(freq_down, spectra_on_down[0])
ax.set_xticks([])
ax = fig.add_subplot(3, 1, 2)
ax.plot(freq_down_off, spectra_off_down[0])
ax.set_xticks([])
ax = fig.add_subplot(3, 1, 3)
ax.plot(freq_down, spectra_on_down[0] - spectra_off_down[0])

This was the first spectrum, but we can now do an average of all spectra:

In [ ]:
spectra_on_down_avg = np.nanmean(np.ma.array(spectra_on_down).filled(np.nan), axis=0)
spectra_off_down_avg = np.nanmean(np.ma.array(spectra_off_down).filled(np.nan), axis=0)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(3, 1, 1)
ax.set_xticks([])
ax.plot(freq_down, spectra_on_down_avg)
ax = fig.add_subplot(3, 1, 2)
ax.plot(freq_down_off, spectra_off_down_avg)
ax.set_xticks([])
ax = fig.add_subplot(3, 1, 3)
ax.plot(freq_down, spectra_on_down_avg - spectra_off_down_avg)